# 🧪 Module 4: Inferential Statistics — Z-Scores, T-Tests & P-Values

**The Story:** PizzaChain has 50 stores. The data team faces three types of questions:

1. **"Is this store abnormal?"** → Z-Score
2. **"Are these two groups actually different?"** → T-Test
3. **"Is this result real or just luck?"** → P-Value

We'll use the **same dataset** throughout so you see how each tool builds on the previous one.

### 🗺️ The Roadmap
```
Z-Score (one value vs population)  →  T-Test (two groups vs each other)  →  P-Value (is it real?)
       "Is Store X weird?"              "Is East ≠ West?"                  "Can I trust this?"
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n = 50

stores = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n)],
    'city': np.random.choice(['New York', 'Chicago', 'LA', 'Houston', 'Phoenix'], n),
    'daily_sales': np.round(np.random.normal(500, 100, n)).astype(int),
    'delivery_time_min': np.round(np.random.normal(32, 8, n), 1),
})

# Assign regions
stores['region'] = stores['city'].apply(lambda c: 'East' if c in ['New York', 'Chicago'] else 'West')

# Make East slightly better for the t-test case
stores.loc[stores['region'] == 'East', 'daily_sales'] += 45

# Add one outlier store
stores.loc[0, 'daily_sales'] = 850
stores.loc[0, 'store_id'] = 'Store_Flagship'

print(f'Dataset: {len(stores)} stores')
print(f'East stores: {(stores["region"]=="East").sum()}, West stores: {(stores["region"]=="West").sum()}')
stores.head(10)

---
## Part 1: Z-Score — "Is This Store Abnormal?"

**Real-life question:** The flagship store made $850 in daily sales. Is that unusually high, or within normal range?

**When to use Z-Score:**
- You have ONE value and want to know how unusual it is compared to the group
- You know (or can estimate) the population mean and std dev
- Examples: Is this server's response time abnormal? Is this student's score exceptional?

**Formula:**
```
z = (your value − mean) / std dev
```

**Interpretation:**
| Z-Score | Meaning | Percentile |
|---|---|---|
| 0 | Exactly average | 50th |
| ±1 | Slightly unusual | 16th or 84th |
| ±2 | Unusual (top/bottom 2.5%) | 2.5th or 97.5th |
| ±3 | Extremely rare (0.1%) | Investigate! |

In [ ]:
# Step-by-step Z-score calculation
mu = stores['daily_sales'].mean()
sigma = stores['daily_sales'].std()
flagship_sales = 850

z = (flagship_sales - mu) / sigma
percentile = stats.norm.cdf(z) * 100

print('📊 Z-SCORE: STEP-BY-STEP MATH')
print('=' * 55)
print(f'  Population mean (μ)     = ${mu:.1f}')
print(f'  Population std dev (σ)  = ${sigma:.1f}')
print(f'  Flagship store sales    = ${flagship_sales}')
print()
print(f'  z = (x − μ) / σ')
print(f'    = ({flagship_sales} − {mu:.1f}) / {sigma:.1f}')
print(f'    = {flagship_sales - mu:.1f} / {sigma:.1f}')
print(f'    = {z:.2f}')
print()
print(f'  Percentile: {percentile:.1f}% (top {100-percentile:.1f}%)')
print()
print(f'💡 INFERENCE:')
if abs(z) > 3:
    print(f'   z = {z:.2f} → EXTREMELY unusual (beyond 3σ).')
    print(f'   Only {100-percentile:.1f}% of stores perform this well.')
    print(f'   This is NOT normal variation — investigate what makes this store special.')
elif abs(z) > 2:
    print(f'   z = {z:.2f} → Unusual (beyond 2σ). Worth investigating.')
else:
    print(f'   z = {z:.2f} → Within normal range. Nothing special.')

In [ ]:
# Visualize: bell curve with the flagship store marked
fig, ax = plt.subplots(figsize=(13, 5))

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
y = stats.norm.pdf(x, mu, sigma)

ax.plot(x, y, color='#7c6aff', linewidth=2.5)
ax.fill_between(x, y, alpha=0.15, color='#7c6aff')

# Shade area beyond flagship
x_beyond = x[x >= flagship_sales]
ax.fill_between(x_beyond, stats.norm.pdf(x_beyond, mu, sigma), color='#f45d6d', alpha=0.4, label=f'Top {100-percentile:.1f}%')

# Mark sigma boundaries
for mult in [1, 2, 3]:
    ax.axvline(mu + mult*sigma, color='#2d3148', linewidth=1, linestyle=':')
    ax.axvline(mu - mult*sigma, color='#2d3148', linewidth=1, linestyle=':')
    ax.text(mu + mult*sigma, max(y)*0.02, f'+{mult}σ', ha='center', fontsize=8, color='#8892b0')

# Mark flagship
ax.axvline(flagship_sales, color='#f45d6d', linewidth=2.5, linestyle='--')
ax.annotate(f'Flagship: ${flagship_sales}\nz = {z:.2f}\nTop {100-percentile:.1f}%',
            xy=(flagship_sales, stats.norm.pdf(flagship_sales, mu, sigma)),
            xytext=(flagship_sales + 30, max(y)*0.6),
            fontsize=11, fontweight='bold', color='#f45d6d',
            arrowprops=dict(arrowstyle='->', color='#f45d6d', lw=2))

ax.axvline(mu, color='#22d3a7', linewidth=2, linestyle='--', label=f'Mean = ${mu:.0f}')

# Plot all stores as dots
for _, row in stores.iterrows():
    c = '#f45d6d' if row['daily_sales'] == 850 else '#22d3a7'
    s = 80 if row['daily_sales'] == 850 else 20
    ax.scatter(row['daily_sales'], -max(y)*0.03, color=c, s=s, zorder=5, alpha=0.7)

ax.set_xlabel('Daily Sales ($)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Z-Score: Where Does the Flagship Store Fall?', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Z-scores for ALL stores — find the unusual ones
stores['z_score'] = (stores['daily_sales'] - mu) / sigma
stores['z_label'] = stores['z_score'].apply(
    lambda z: '🔴 Very unusual' if abs(z) > 2.5 else '🟡 Unusual' if abs(z) > 2 else '🟢 Normal')

print('🚨 STORES WITH UNUSUAL SALES (|z| > 2):')
unusual = stores[stores['z_score'].abs() > 2][['store_id', 'city', 'daily_sales', 'z_score', 'z_label']]
print(unusual.to_string(index=False) if len(unusual) > 0 else '  All stores are within normal range!')
print()
print('💡 Z-scores let you compare apples to oranges.')
print('   A $700 store in a chain averaging $500 (z=2.0) is MORE unusual')
print('   than a $900 store in a chain averaging $800 (z=1.0).')

---
## Part 2: T-Test — "Are East and West Stores Actually Different?"

**Real-life question:** East region stores seem to sell more than West. But is the difference REAL, or just random variation?

**When to use T-Test:**
- You have TWO groups and want to know if their averages are truly different
- You DON'T know the population std dev (you estimate it from the sample)
- Examples: A/B test results, comparing regions, before vs after a change

**Three types:**
| Type | Question | Example |
|---|---|---|
| One-sample | Is this group different from a known value? | "Is our avg delivery time ≠ 30 min SLA?" |
| Two-sample (independent) | Are two separate groups different? | "East vs West sales" |
| Paired | Did the SAME group change? | "Same stores before vs after renovation" |

**Formula (two-sample):**
```
t = (mean₁ − mean₂) / √(s₁²/n₁ + s₂²/n₂)
```
Where s = sample std dev, n = sample size

**Z-Test vs T-Test:**
| | Z-Test | T-Test |
|---|---|---|
| Know population σ? | Yes | No (estimate from sample) |
| Sample size | Large (n > 30) | Any size |
| In practice | Rare | Used 99% of the time |

In [ ]:
east = stores[stores['region'] == 'East']['daily_sales']
west = stores[stores['region'] == 'West']['daily_sales']

print('📊 DATA COMPARISON')
print('=' * 55)
print(f'  East: n={len(east)}, mean=${east.mean():.1f}, std=${east.std():.1f}')
print(f'  West: n={len(west)}, mean=${west.mean():.1f}, std=${west.std():.1f}')
print(f'  Difference: ${east.mean() - west.mean():.1f}')
print()
print('  East looks higher... but is it REAL or just luck?')

In [ ]:
# Step-by-step t-test calculation
mean_diff = east.mean() - west.mean()
se = np.sqrt(east.var()/len(east) + west.var()/len(west))  # standard error
t_stat = mean_diff / se

# Using scipy for the official result
t_stat_scipy, p_value = stats.ttest_ind(east, west)

print('📊 T-TEST: STEP-BY-STEP MATH')
print('=' * 55)
print(f'  Step 1: Difference in means')
print(f'    mean_east − mean_west = {east.mean():.1f} − {west.mean():.1f} = {mean_diff:.1f}')
print()
print(f'  Step 2: Standard Error (how much could this differ by chance?)')
print(f'    SE = √(s₁²/n₁ + s₂²/n₂)')
print(f'       = √({east.var():.1f}/{len(east)} + {west.var():.1f}/{len(west)})')
print(f'       = √({east.var()/len(east):.1f} + {west.var()/len(west):.1f})')
print(f'       = {se:.2f}')
print()
print(f'  Step 3: T-statistic (signal / noise)')
print(f'    t = difference / SE = {mean_diff:.1f} / {se:.2f} = {t_stat:.2f}')
print()
print(f'  Step 4: P-value (from scipy)')
print(f'    t = {t_stat_scipy:.4f}, p = {p_value:.4f}')
print()
print(f'💡 INFERENCE:')
if p_value < 0.05:
    print(f'   p = {p_value:.4f} < 0.05 → SIGNIFICANT! ✅')
    print(f'   The ${mean_diff:.0f} difference is real, not random noise.')
    print(f'   East stores genuinely outperform West stores.')
    print(f'   ACTION: Investigate what East is doing differently.')
else:
    print(f'   p = {p_value:.4f} > 0.05 → NOT significant ❌')
    print(f'   The difference could be random variation.')
    print(f'   We cannot conclude East is truly better.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: overlapping histograms
axes[0].hist(east, bins=12, alpha=0.6, color='#7c6aff', label=f'East (mean=${east.mean():.0f})', edgecolor='white')
axes[0].hist(west, bins=12, alpha=0.6, color='#22d3a7', label=f'West (mean=${west.mean():.0f})', edgecolor='white')
axes[0].axvline(east.mean(), color='#7c6aff', linewidth=2, linestyle='--')
axes[0].axvline(west.mean(), color='#22d3a7', linewidth=2, linestyle='--')
axes[0].set_xlabel('Daily Sales ($)')
axes[0].set_title('East vs West: Distribution Overlap', fontsize=12, fontweight='bold')
axes[0].legend()

# Right: t-distribution with our t-stat marked
df_val = len(east) + len(west) - 2
x_t = np.linspace(-4, 4, 300)
y_t = stats.t.pdf(x_t, df_val)
axes[1].plot(x_t, y_t, color='#7c6aff', linewidth=2)
axes[1].fill_between(x_t, y_t, alpha=0.1, color='#7c6aff')

# Shade rejection regions
crit = stats.t.ppf(0.975, df_val)
x_reject_r = x_t[x_t >= crit]
x_reject_l = x_t[x_t <= -crit]
axes[1].fill_between(x_reject_r, stats.t.pdf(x_reject_r, df_val), color='#f45d6d', alpha=0.4, label='Rejection zone (p<0.05)')
axes[1].fill_between(x_reject_l, stats.t.pdf(x_reject_l, df_val), color='#f45d6d', alpha=0.4)

# Mark our t-stat
axes[1].axvline(t_stat_scipy, color='#f5b731', linewidth=2.5, linestyle='--', label=f'Our t = {t_stat_scipy:.2f}')
axes[1].set_xlabel('T-statistic')
axes[1].set_title(f'T-Distribution: Is Our t={t_stat_scipy:.2f} in the Rejection Zone?', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

in_zone = 'YES — in the red zone' if abs(t_stat_scipy) > crit else 'NO — in the safe zone'
print(f'💡 Critical value = ±{crit:.2f}. Our t = {t_stat_scipy:.2f}. {in_zone}.')

---
### Case 2b: One-Sample T-Test — "Is our delivery time meeting the 30-min SLA?"

**Question:** Management promised customers a 30-minute average delivery. Are we meeting it?

In [ ]:
target_sla = 30  # minutes
delivery = stores['delivery_time_min']

t_one, p_one = stats.ttest_1samp(delivery, target_sla)

print('📊 ONE-SAMPLE T-TEST: STEP-BY-STEP')
print('=' * 55)
print(f'  SLA target         = {target_sla} min')
print(f'  Our sample mean    = {delivery.mean():.1f} min')
print(f'  Our sample std     = {delivery.std():.1f} min')
print(f'  Sample size        = {len(delivery)}')
print()
print(f'  t = (sample_mean − target) / (s / √n)')
print(f'    = ({delivery.mean():.1f} − {target_sla}) / ({delivery.std():.1f} / √{len(delivery)})')
print(f'    = {delivery.mean() - target_sla:.1f} / {delivery.std()/np.sqrt(len(delivery)):.2f}')
print(f'    = {t_one:.2f}')
print(f'  p-value = {p_one:.4f}')
print()
if p_one < 0.05:
    direction = 'ABOVE' if delivery.mean() > target_sla else 'BELOW'
    print(f'💡 INFERENCE: p={p_one:.4f} < 0.05 → Our delivery time is significantly {direction} the SLA.')
    print(f'   The {abs(delivery.mean()-target_sla):.1f} min difference is NOT random — it\'s a real problem (or win).')
else:
    print(f'💡 INFERENCE: p={p_one:.4f} > 0.05 → We cannot say we\'re different from the 30-min target.')
    print(f'   We\'re meeting the SLA (statistically speaking).')

---
### Case 2c: Paired T-Test — "Did the new oven improve delivery times?"

**Question:** 20 stores got new ovens. Did their delivery times improve? We compare the SAME stores before and after.

In [ ]:
n_paired = 20
np.random.seed(99)
before = np.round(np.random.normal(35, 6, n_paired), 1)
after = before - np.random.normal(3, 4, n_paired)  # on average 3 min faster
after = np.round(after, 1)

diff = before - after  # positive = improvement
t_paired, p_paired = stats.ttest_rel(before, after)

print('📊 PAIRED T-TEST: STEP-BY-STEP')
print('=' * 55)
print(f'  Stores tested      = {n_paired}')
print(f'  Mean BEFORE        = {before.mean():.1f} min')
print(f'  Mean AFTER         = {after.mean():.1f} min')
print(f'  Mean improvement   = {diff.mean():.1f} min')
print()
print(f'  The paired t-test looks at the DIFFERENCES for each store:')
print(f'  Mean diff = {diff.mean():.2f}, Std of diffs = {diff.std():.2f}')
print(f'  t = mean_diff / (std_diff / √n) = {diff.mean():.2f} / ({diff.std():.2f}/√{n_paired}) = {t_paired:.2f}')
print(f'  p-value = {p_paired:.4f}')
print()
if p_paired < 0.05:
    print(f'💡 INFERENCE: p={p_paired:.4f} < 0.05 → The new ovens DID improve delivery times! ✅')
    print(f'   Average improvement: {diff.mean():.1f} minutes per store.')
    print(f'   ACTION: Roll out new ovens to all stores.')
else:
    print(f'💡 INFERENCE: p={p_paired:.4f} > 0.05 → No significant improvement detected.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: before vs after for each store
store_nums = range(1, n_paired + 1)
axes[0].scatter(store_nums, before, color='#f45d6d', s=60, label='Before', zorder=3)
axes[0].scatter(store_nums, after, color='#22d3a7', s=60, label='After', zorder=3)
for i in range(n_paired):
    color = '#22d3a7' if after[i] < before[i] else '#f45d6d'
    axes[0].plot([store_nums[i], store_nums[i]], [before[i], after[i]], color=color, linewidth=1.5, alpha=0.6)
axes[0].set_xlabel('Store #')
axes[0].set_ylabel('Delivery Time (min)')
axes[0].set_title('Before vs After: Each Store', fontsize=12, fontweight='bold')
axes[0].legend()

# Right: distribution of differences
colors_diff = ['#22d3a7' if d > 0 else '#f45d6d' for d in diff]
axes[1].bar(store_nums, diff, color=colors_diff, alpha=0.7, edgecolor='white')
axes[1].axhline(0, color='white', linewidth=1)
axes[1].axhline(diff.mean(), color='#f5b731', linewidth=2, linestyle='--', label=f'Mean improvement: {diff.mean():.1f} min')
axes[1].set_xlabel('Store #')
axes[1].set_ylabel('Improvement (min)')
axes[1].set_title('Improvement per Store (green = faster)', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 3: P-Value — "Can I Trust This Result?"

**The p-value ties everything together.** Both Z-scores and T-tests produce a p-value.

**What it means:**
```
p-value = "If there's truly NO difference (H₀ is true),
           what's the probability of seeing a result THIS extreme?"
```

**Decision rule:**
- p < 0.05 → Reject H₀ → "The result is statistically significant"
- p ≥ 0.05 → Fail to reject H₀ → "Could be random noise"

**⚠️ Common mistakes:**
- p = 0.03 does NOT mean "3% chance the null is true"
- p < 0.05 does NOT mean the effect is large or important
- Statistical significance ≠ practical significance

In [ ]:
# Let's see p-values in action with a concrete A/B test
# Scenario: PizzaChain tests a new menu layout in 25 stores

np.random.seed(42)
control_sales = np.random.normal(500, 90, 25)   # old menu
test_sales = np.random.normal(530, 90, 25)       # new menu (+$30 real effect)

t_ab, p_ab = stats.ttest_ind(control_sales, test_sales)

print('📊 A/B TEST: NEW MENU LAYOUT')
print('=' * 55)
print(f'  Control (old menu): n={len(control_sales)}, mean=${control_sales.mean():.1f}, std=${control_sales.std():.1f}')
print(f'  Test (new menu):    n={len(test_sales)}, mean=${test_sales.mean():.1f}, std=${test_sales.std():.1f}')
print(f'  Difference:         ${test_sales.mean() - control_sales.mean():.1f}')
print(f'  T-statistic:        {t_ab:.2f}')
print(f'  P-value:            {p_ab:.4f}')
print()
if p_ab < 0.05:
    print(f'  ✅ p = {p_ab:.4f} < 0.05 → SIGNIFICANT!')
    print(f'  The new menu genuinely increases sales by ~${test_sales.mean()-control_sales.mean():.0f}/day.')
    print(f'  Across 50 stores, that\'s ~${(test_sales.mean()-control_sales.mean())*50:.0f}/day extra revenue.')
else:
    print(f'  ❌ p = {p_ab:.4f} ≥ 0.05 → Not significant.')
    print(f'  The difference could be random. Need more stores or a bigger effect.')

In [ ]:
# Visualize: what the p-value actually represents
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: the two groups
axes[0].hist(control_sales, bins=10, alpha=0.6, color='#7c6aff', label=f'Control (${control_sales.mean():.0f})', edgecolor='white')
axes[0].hist(test_sales, bins=10, alpha=0.6, color='#22d3a7', label=f'Test (${test_sales.mean():.0f})', edgecolor='white')
axes[0].axvline(control_sales.mean(), color='#7c6aff', linewidth=2, linestyle='--')
axes[0].axvline(test_sales.mean(), color='#22d3a7', linewidth=2, linestyle='--')
axes[0].set_title('Control vs Test Group', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Daily Sales ($)')
axes[0].legend()

# Right: p-value on the t-distribution
df_ab = len(control_sales) + len(test_sales) - 2
x_t = np.linspace(-4, 4, 300)
y_t = stats.t.pdf(x_t, df_ab)
axes[1].plot(x_t, y_t, color='#7c6aff', linewidth=2)
axes[1].fill_between(x_t, y_t, alpha=0.1, color='#7c6aff')

# Shade p-value area (two-tailed)
x_right = x_t[x_t >= abs(t_ab)]
x_left = x_t[x_t <= -abs(t_ab)]
axes[1].fill_between(x_right, stats.t.pdf(x_right, df_ab), color='#f45d6d', alpha=0.5, label=f'p-value = {p_ab:.4f}')
axes[1].fill_between(x_left, stats.t.pdf(x_left, df_ab), color='#f45d6d', alpha=0.5)
axes[1].axvline(t_ab, color='#f5b731', linewidth=2.5, linestyle='--', label=f't = {t_ab:.2f}')

axes[1].set_title('P-Value = Red Shaded Area', fontsize=12, fontweight='bold')
axes[1].set_xlabel('T-statistic')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print('💡 The red area IS the p-value — the probability of seeing a result')
print('   this extreme if there were truly no difference between the groups.')
print(f'   Our red area = {p_ab:.4f} = {p_ab*100:.2f}% — {"small enough to be confident!" if p_ab < 0.05 else "too large to be sure."}')

---
### Bonus: Effect Size — "OK it's significant, but is it MEANINGFUL?"

**Cohen's d** measures the SIZE of the difference in standard deviation units:
```
d = (mean₁ − mean₂) / pooled_std
```
| d | Interpretation |
|---|---|
| 0.2 | Small effect |
| 0.5 | Medium effect |
| 0.8+ | Large effect |

In [ ]:
pooled_std = np.sqrt((control_sales.std()**2 + test_sales.std()**2) / 2)
cohens_d = (test_sales.mean() - control_sales.mean()) / pooled_std

print(f'📊 EFFECT SIZE')
print(f'  Cohen\'s d = ({test_sales.mean():.1f} − {control_sales.mean():.1f}) / {pooled_std:.1f} = {cohens_d:.2f}')
print()
size = 'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'
print(f'  Interpretation: {size} effect')
print()
print(f'💡 INFERENCE:')
print(f'   The p-value says the difference is REAL (not luck).')
print(f'   Cohen\'s d says the difference is {size.lower()} in practical terms.')
print(f'   BOTH matter: a tiny real difference might not be worth acting on.')

---
## 📝 Summary: When to Use What

| Tool | Question | Input | Output | Example |
|---|---|---|---|---|
| **Z-Score** | Is this ONE value unusual? | One value + population μ,σ | z-score + percentile | "Is Store X abnormal?" |
| **One-sample t-test** | Is this group ≠ a target? | One group + target value | t-stat + p-value | "Are we meeting the 30-min SLA?" |
| **Two-sample t-test** | Are two groups different? | Two groups | t-stat + p-value | "East vs West sales" |
| **Paired t-test** | Did the same group change? | Before + After (same subjects) | t-stat + p-value | "Did new ovens help?" |
| **P-value** | Is the result real? | Any test statistic | Probability | "Can I trust this?" |
| **Cohen's d** | Is the effect meaningful? | Two group means + std | Effect size | "Is it worth acting on?" |

### 🔑 The Decision Flow:
```
1. Is ONE value weird?           → Z-Score
2. Is a GROUP different from X?   → One-sample t-test
3. Are TWO GROUPS different?      → Two-sample t-test
4. Did the SAME group change?     → Paired t-test
5. Check p-value < 0.05?          → Statistically significant
6. Check Cohen's d > 0.5?         → Practically meaningful
```